# Fase 1 — Reconocimiento de Patrones: Firmas en Cheques Off-Line

**Descripción general**

La verificación de firmas *off-line* trabaja sobre imágenes de cheques en papel. A diferencia de la verificación *on-line*, la información dinámica del trazo se pierde, por lo que el sistema debe apoyarse en características morfológicas extraídas de la imagen.

El flujo de trabajo completo es:

```
Cheque real (imagen completa)
        │
        ▼
  ┌─────────────┐
  │  U-Net      │  ← entrenada con pares (X_cheque, y_firma)
  │  Segmentación│
  └──────┬──────┘
         │  Máscara de firma
         ▼
  Post-procesamiento (binarización, limpieza, recorte)
         │
         ▼
  Extracción de características (54 EE morfológicos)
         │
         ▼
  Clasificadores (BPNN · SVM · KNN · Naive Bayes)
         │
         ▼
  Auténtica / Falsificación
```

---

## 0 · Dependencias

In [ ]:
# ── Instalación (ejecutar sólo la primera vez) ────────────────────────────────
# !pip install torch torchvision numpy matplotlib scikit-image scikit-learn Pillow tqdm

import os, re, random, warnings
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import tqdm
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from skimage import filters, morphology
from skimage.morphology import binary_erosion, binary_opening
from scipy.ndimage import rotate as ndrotate

from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc
)
from sklearn.model_selection import cross_val_score

warnings.filterwarnings('ignore')
np.random.seed(42); random.seed(42); torch.manual_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✔  Dependencias cargadas.  Dispositivo: {DEVICE}')

---
## Etapa 1 · Selección de Datos
### Adquisición y carga de imágenes de cheques con firmas off-line

El conjunto de datos tiene la siguiente estructura:
```
dataset/
├── TrainSet/
│   ├── X/   ← cheque completo         (X_<n>.png)
│   └── y/   ← firma aislada, negativo (y_<n>.png)
└── TestSet/
    ├── X/
    └── y/
```
Los pares `(X_n, y_n)` se usan para entrenar la U-Net. Las imágenes en `y` son el **ground truth de segmentación**: la firma aislada en negativo (trazo oscuro sobre fondo claro, o viceversa según el dataset). La red aprende a localizar y extraer la firma desde el cheque completo.

In [ ]:
# ── Configuración de rutas ────────────────────────────────────────────────────
DATASET_ROOT = './'   # ← ajusta a tu ruta

PATHS = {
    'train_X': os.path.join(DATASET_ROOT, 'TrainSet', 'X'),
    'train_y': os.path.join(DATASET_ROOT, 'TrainSet', 'y'),
    'test_X' : os.path.join(DATASET_ROOT, 'TestSet',  'X'),
    'test_y' : os.path.join(DATASET_ROOT, 'TestSet',  'y'),
}

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')

def sorted_images(folder):
    files = [f for f in os.listdir(folder) if f.lower().endswith(IMG_EXTS)]
    files.sort(key=lambda f: int(re.search(r'(\d+)', f).group()))
    return [os.path.join(folder, f) for f in files]

train_X_paths = sorted_images(PATHS['train_X'])
train_y_paths = sorted_images(PATHS['train_y'])
test_X_paths  = sorted_images(PATHS['test_X'])
test_y_paths  = sorted_images(PATHS['test_y'])

assert len(train_X_paths) == len(train_y_paths), 'Pares train no coinciden'
assert len(test_X_paths)  == len(test_y_paths),  'Pares test no coinciden'

print(f'Pares de entrenamiento : {len(train_X_paths)}')
print(f'Pares de prueba        : {len(test_X_paths)}')

In [ ]:
# ── Fig. 1 · Mosaico de la base de datos ─────────────────────────────────────
def plot_mosaic(paths, title, cols=5, max_imgs=20):
    paths = paths[:max_imgs]
    rows  = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*2.2))
    axes = np.array(axes).flatten()
    for ax, p in zip(axes, paths):
        ax.imshow(np.array(Image.open(p).convert('L')), cmap='gray')
        ax.set_title(os.path.basename(p), fontsize=7)
        ax.axis('off')
    for ax in axes[len(paths):]: ax.axis('off')
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout(); plt.show()

plot_mosaic(train_X_paths, 'Fig. 1a · Base de Datos — Cheques Completos (TrainSet)')
plot_mosaic(train_y_paths, 'Fig. 1b · Base de Datos — Firmas / Ground Truth (TrainSet)')

In [ ]:
# ── Fig. 2 · Par cheque – firma (ejemplo) ────────────────────────────────────
idx = 0
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(np.array(Image.open(train_X_paths[idx]).convert('RGB')))
axes[0].set_title(f'Fig. 2a · Cheque: {os.path.basename(train_X_paths[idx])}', fontsize=10)
axes[0].axis('off')
axes[1].imshow(np.array(Image.open(train_y_paths[idx]).convert('L')), cmap='gray')
axes[1].set_title(f'Fig. 2b · Firma GT: {os.path.basename(train_y_paths[idx])}', fontsize=10)
axes[1].axis('off')
plt.suptitle('Fig. 2 · Adquisición de la firma en imagen de cheque', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Etapa 2 · Pre-procesamiento de Datos
### Red de Segmentación U-Net para Extracción Automática de la Firma

El pre-procesamiento tiene **dos sub-fases**:

**2A — Entrenamiento de la U-Net de segmentación**  
Se entrena una U-Net ligera con los pares `(cheque completo, máscara de firma)`.  
La red aprende a localizar la región de firma dentro del cheque, independientemente del diseño del mismo.

**2B — Post-procesamiento de la máscara predicha**  
La máscara predicha se umbraliza, se limpia y se recorta al bounding-box de la firma.
A partir de este punto, **todo el pipeline trabaja sobre la firma extraída automáticamente**.

In [ ]:
# ── 2A · Dataset PyTorch ──────────────────────────────────────────────────────
UNET_SIZE = (256, 512)   # (H, W) — múltiplo de 32

class CheckSignatureDataset(Dataset):
    """
    Carga pares (cheque, máscara_firma).
    Cheque  → tensor float32 [1, H, W] normalizado [0,1]
    Máscara → tensor float32 [1, H, W] binaria {0, 1}  (firma=1)
    """
    def __init__(self, x_paths, y_paths, size=UNET_SIZE, augment=False):
        self.x_paths = x_paths
        self.y_paths = y_paths
        self.size    = size
        self.augment = augment

    def __len__(self): return len(self.x_paths)

    def __getitem__(self, idx):
        # Cheque en escala de grises
        img  = Image.open(self.x_paths[idx]).convert('L')
        img  = img.resize((self.size[1], self.size[0]), Image.LANCZOS)
        img  = np.array(img, dtype=np.float32) / 255.0

        # Máscara de firma
        mask = Image.open(self.y_paths[idx]).convert('L')
        mask = mask.resize((self.size[1], self.size[0]), Image.NEAREST)
        mask = np.array(mask, dtype=np.float32)
        thresh = filters.threshold_otsu(mask) if mask.std() > 1 else 127.0
        mask   = (mask < thresh).astype(np.float32)   # firma = 1

        # Augmentation: volteo horizontal
        if self.augment and random.random() > 0.5:
            img  = img[:,  ::-1].copy()
            mask = mask[:, ::-1].copy()

        return (torch.from_numpy(img).unsqueeze(0),
                torch.from_numpy(mask).unsqueeze(0))


train_ds = CheckSignatureDataset(train_X_paths, train_y_paths, augment=True)
test_ds  = CheckSignatureDataset(test_X_paths,  test_y_paths,  augment=False)
train_dl = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=0)
test_dl  = DataLoader(test_ds,  batch_size=4, shuffle=False, num_workers=0)

xb, yb = next(iter(train_dl))
print(f'Tensor cheque : {xb.shape}  |  Tensor máscara: {yb.shape}')

In [ ]:
# ── 2A · Arquitectura U-Net ligera ───────────────────────────────────────────
#
# Encoder : 4 × (DoubleConv + MaxPool)
# Bottleneck: DoubleConv
# Decoder : 4 × (ConvTranspose + skip-connection + DoubleConv)
# Output  : Conv 1×1 + Sigmoid → máscara binaria

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class UNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        b = base
        self.enc1 = DoubleConv(1,    b);    self.pool = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(b,    b*2)
        self.enc3 = DoubleConv(b*2,  b*4)
        self.enc4 = DoubleConv(b*4,  b*8)
        self.bot  = DoubleConv(b*8,  b*16)
        self.up4  = nn.ConvTranspose2d(b*16, b*8,  2, stride=2)
        self.dec4 = DoubleConv(b*16, b*8)
        self.up3  = nn.ConvTranspose2d(b*8,  b*4,  2, stride=2)
        self.dec3 = DoubleConv(b*8,  b*4)
        self.up2  = nn.ConvTranspose2d(b*4,  b*2,  2, stride=2)
        self.dec2 = DoubleConv(b*4,  b*2)
        self.up1  = nn.ConvTranspose2d(b*2,  b,    2, stride=2)
        self.dec1 = DoubleConv(b*2,  b)
        self.out  = nn.Conv2d(b, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bot(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b),  e4], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return torch.sigmoid(self.out(d1))


model = UNet(base=32).to(DEVICE)
print(f'U-Net inicializada — {sum(p.numel() for p in model.parameters()):,} parámetros')

In [ ]:
# ── 2A · Función de pérdida: BCE + Dice ──────────────────────────────────────
# La combinación es más robusta ante desequilibrio de clases
# (la firma ocupa solo una fracción pequeña del cheque).

def dice_loss(pred, target, smooth=1.):
    p = pred.contiguous().view(-1)
    t = target.contiguous().view(-1)
    return 1 - (2*(p*t).sum() + smooth) / (p.sum() + t.sum() + smooth)

def bce_dice(pred, target):
    return F.binary_cross_entropy(pred, target) + dice_loss(pred, target)

def dice_score(pred_bin, target):
    p = pred_bin.contiguous().view(-1).float()
    t = target.contiguous().view(-1).float()
    return (2*(p*t).sum() + 1.) / (p.sum() + t.sum() + 1.)

print('Funciones de pérdida definidas ✔')

In [ ]:
for epoch in range(1, EPOCHS+1):
    model.train()
    tl, td = 0., 0.
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        pred = model(xb)
        loss = bce_dice(pred, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tl += loss.item()
        td += dice_score((pred > THRESHOLD).float(), yb).item()
    tl /= len(train_dl); td /= len(train_dl)

    model.eval()
    vl, vd = 0., 0.
    with torch.no_grad():
        for xb, yb in test_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb)
            vl += bce_dice(pred, yb).item()
            vd += dice_score((pred > THRESHOLD).float(), yb).item()
    vl /= len(test_dl); vd /= len(test_dl)

    # ── Scheduler con log manual ──────────────────────────────
    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(vl)
    new_lr = optimizer.param_groups[0]['lr']
    if new_lr < old_lr:
        print(f'  ↓ LR reducido: {old_lr:.2e} → {new_lr:.2e}')
    # ─────────────────────────────────────────────────────────

    for k, v in zip(['train_loss','train_dice','val_loss','val_dice'], [tl,td,vl,vd]):
        history[k].append(v)

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS}  '
              f'Train loss={tl:.4f} dice={td:.4f}  |  '
              f'Val loss={vl:.4f} dice={vd:.4f}')

print('\n✔  Entrenamiento finalizado.')

In [ ]:
# ── Fig. 3 · Curvas de entrenamiento ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (key_tr, key_val, ylabel) in zip(axes, [
    ('train_loss','val_loss','Loss (BCE+Dice)'),
    ('train_dice','val_dice','Coeficiente Dice')
]):
    ax.plot(history[key_tr],  label='Train', color='steelblue')
    ax.plot(history[key_val], label='Val',   color='tomato', linestyle='--')
    ax.set_xlabel('Época'); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('Fig. 3 · Curvas de entrenamiento — U-Net de segmentación',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Dice final (val): {history["val_dice"][-1]:.4f}')

In [ ]:
# ── Guardar modelo ────────────────────────────────────────────────────────────
MODEL_PATH = 'unet_signature.pth'
torch.save(model.state_dict(), MODEL_PATH)
print(f'Modelo guardado en: {MODEL_PATH}')
# Para cargar: model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

In [ ]:
# ── 2B · Función principal: cheque → firma binaria limpia ────────────────────
TARGET_SIG_SIZE = (128, 256)   # tamaño fijo para los clasificadores


def extract_signature_from_check(img_path: str,
                                  model: nn.Module,
                                  unet_size  = UNET_SIZE,
                                  sig_size   = TARGET_SIG_SIZE,
                                  threshold  = THRESHOLD) -> np.ndarray:
    """
    Dado el path de un cheque, devuelve la firma extraída como
    array booleano de forma sig_size (alto, ancho).

    Pasos:
    1. Redimensionar y normalizar la imagen del cheque.
    2. Predecir máscara con la U-Net.
    3. Umbralizar y limpiar con apertura morfológica.
    4. Recortar al bounding-box de la región detectada.
    5. Redimensionar a sig_size.
    """
    model.eval()

    # 1. Preparar imagen
    img    = Image.open(img_path).convert('L')
    img_r  = img.resize((unet_size[1], unet_size[0]), Image.LANCZOS)
    img_np = np.array(img_r, dtype=np.float32) / 255.0
    img_t  = torch.from_numpy(img_np).unsqueeze(0).unsqueeze(0).to(DEVICE)

    # 2. Predicción
    with torch.no_grad():
        pred_mask = model(img_t).squeeze().cpu().numpy()   # (H, W) en [0,1]

    # 3. Binarizar y limpiar
    binary  = pred_mask > threshold
    cleaned = binary_opening(binary, morphology.disk(2))

    # 4. Bounding-box
    rows, cols = np.any(cleaned, axis=1), np.any(cleaned, axis=0)
    if rows.any() and cols.any():
        r0, r1 = np.where(rows)[0][[0, -1]]
        c0, c1 = np.where(cols)[0][[0, -1]]
        pad = 5
        r0 = max(0, r0-pad); r1 = min(cleaned.shape[0]-1, r1+pad)
        c0 = max(0, c0-pad); c1 = min(cleaned.shape[1]-1, c1+pad)
        crop = cleaned[r0:r1+1, c0:c1+1]
    else:
        crop = cleaned

    # 5. Redimensionar a tamaño fijo
    pil = Image.fromarray(crop.astype(np.uint8)*255)
    pil = pil.resize((sig_size[1], sig_size[0]), Image.LANCZOS)
    return np.array(pil) > 127


print('extract_signature_from_check() definida ✔')

In [ ]:
# ── Fig. 4 · Pipeline completo de extracción (visualización) ─────────────────
def visualize_pipeline(check_path, gt_path, model):
    model.eval()
    img   = Image.open(check_path).convert('L')
    img_r = img.resize((UNET_SIZE[1], UNET_SIZE[0]), Image.LANCZOS)
    img_np = np.array(img_r, dtype=np.float32) / 255.0
    img_t  = torch.from_numpy(img_np).unsqueeze(0).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = model(img_t).squeeze().cpu().numpy()

    binary  = pred > THRESHOLD
    cleaned = binary_opening(binary, morphology.disk(2))

    gt_arr   = np.array(Image.open(gt_path).convert('L').resize(
                   (UNET_SIZE[1], UNET_SIZE[0]), Image.NEAREST))
    gt_thresh = filters.threshold_otsu(gt_arr) if gt_arr.std() > 1 else 127
    gt_bin    = gt_arr < gt_thresh

    sig_final = extract_signature_from_check(check_path, model)

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    for ax, (im, cm, t) in zip(axes, [
        (img_np,    'gray', 'a) Cheque (gris)'),
        (gt_bin,    'gray', 'b) GT firma (y_n)'),
        (pred,      'hot',  'c) Prob. U-Net'),
        (cleaned,   'gray', 'd) Máscara binarizada'),
        (sig_final, 'gray', 'e) Firma final (recortada)'),
    ]):
        ax.imshow(im, cmap=cm); ax.set_title(t, fontsize=9); ax.axis('off')

    plt.suptitle('Fig. 4 · Pipeline de extracción automática de firma desde cheque real',
                 fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

visualize_pipeline(train_X_paths[0], train_y_paths[0], model)

In [ ]:
# ── Extraer todas las firmas del dataset ─────────────────────────────────────
print('Extrayendo firmas — TrainSet...')
train_sigs = [extract_signature_from_check(p, model)
              for p in tqdm(train_X_paths, desc='TrainSet')]

print('Extrayendo firmas — TestSet...')
test_sigs  = [extract_signature_from_check(p, model)
              for p in tqdm(test_X_paths,  desc='TestSet')]

print(f'\n✔  {len(train_sigs)} firmas de entrenamiento extraídas.')
print(f'✔  {len(test_sigs)}  firmas de prueba extraídas.')

In [ ]:
# ── Fig. 5 · Comparación GT vs Firma extraída por U-Net ──────────────────────
n = min(4, len(train_sigs))
fig, axes = plt.subplots(2, n, figsize=(n*4, 5))
for i in range(n):
    gt = np.array(Image.open(train_y_paths[i]).convert('L'))
    gt_bin = gt < (filters.threshold_otsu(gt) if gt.std()>1 else 127)
    axes[0,i].imshow(gt_bin, cmap='gray')
    axes[0,i].set_title(f'GT  {os.path.basename(train_y_paths[i])}', fontsize=8)
    axes[0,i].axis('off')
    axes[1,i].imshow(train_sigs[i], cmap='gray')
    axes[1,i].set_title(f'U-Net  {os.path.basename(train_X_paths[i])}', fontsize=8)
    axes[1,i].axis('off')
plt.suptitle('Fig. 5 · Ground Truth vs Firma extraída automáticamente',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Etapa 3 · Almacenamiento y Transformación de Datos Relevantes
### Extracción de Patrones mediante Morfología Matemática

La operación morfológica utilizada es la **erosión binaria**. Se aplica con **54 elementos estructurantes** (EE):

- **36 EE para trazos curvos** — arcos en distintas orientaciones y radios (Fig. 6).
- **18 EE para trazos rectos** — segmentos rectos a ángulos de 0° a 170° cada 10° (Fig. 7).

El número de píxeles encendidos tras la erosión con el EE $k$ forma la componente $k$ del **vector de características de 54 dimensiones**.

In [ ]:
# ── Definición de los 54 EE ───────────────────────────────────────────────────
def make_arc(radius=3, arc_type='c_right'):
    s  = 2*radius+1
    se = np.zeros((s,s), dtype=bool)
    cx = cy = radius
    for y in range(s):
        for x in range(s):
            if abs(np.sqrt((x-cx)**2+(y-cy)**2) - radius) < 1.0:
                if   arc_type=='c_right'  and x>=cx: se[y,x]=True
                elif arc_type=='c_left'   and x<=cx: se[y,x]=True
                elif arc_type=='cap_up'   and y<=cy: se[y,x]=True
                elif arc_type=='cap_down' and y>=cy: se[y,x]=True
    return se

def make_line(angle_deg, length=7):
    se = np.zeros((length,length), dtype=bool)
    se[length//2, :] = True
    return ndrotate(se.astype(float), angle_deg, reshape=False, order=0) > 0.5

# 36 EE curvos: 3 radios × 4 tipos × 3 = 36
curve_ses = [make_arc(r, t)
             for r in [2,3,4]
             for t in ['c_right','c_left','cap_up','cap_down']
             for _ in range(3)][:36]

# 18 EE rectos: 0°,10°,...,170°
straight_ses = [make_line(a) for a in range(0, 180, 10)]

ALL_SES     = curve_ses + straight_ses
TIPO_LABELS = ['Curva']*36 + ['Recta']*18
assert len(ALL_SES) == 54
print(f'Total EE: {len(ALL_SES)}  (curvas={len(curve_ses)}, rectas={len(straight_ses)})')

In [ ]:
# ── Fig. 6-7 · Visualización de los 54 EE ────────────────────────────────────
fig, axes = plt.subplots(6, 9, figsize=(18, 13))
for i, (ax, se) in enumerate(zip(axes.flatten(), ALL_SES)):
    ax.imshow(se, cmap='binary', vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(f'EE{i+1}\n{TIPO_LABELS[i]}', fontsize=6)
    ax.axis('off')
for ax in axes.flatten()[54:]: ax.axis('off')
plt.suptitle('Fig. 6-7 · Los 54 Elementos Estructurantes (36 curvas + 18 rectas)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ── Extracción de características ─────────────────────────────────────────────
def extract_features(sig, ses):
    return np.array([binary_erosion(sig, se).sum() for se in ses], dtype=np.float64)

def build_feature_matrix(sigs, ses):
    return np.vstack([extract_features(s, ses)
                      for s in tqdm(sigs, desc='Extrayendo EE')])

X_train_real = build_feature_matrix(train_sigs, ALL_SES)
X_test_real  = build_feature_matrix(test_sigs,  ALL_SES)
print(f'Matrices — Train: {X_train_real.shape}  |  Test: {X_test_real.shape}')

In [ ]:
# ── Fig. 8 · Resultado de la erosión con los primeros 9 EE ───────────────────
sig_ex = train_sigs[0]
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for i, ax in enumerate(axes.flatten()):
    er = binary_erosion(sig_ex, ALL_SES[i])
    ax.imshow(er, cmap='gray')
    ax.set_title(f'EE{i+1} ({TIPO_LABELS[i]})\n{int(er.sum())} px', fontsize=9)
    ax.axis('off')
plt.suptitle('Fig. 8 · Resultado de erosión con primeros 9 EE',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Generación de patrones sintéticos (Tabla 1) ──────────────────────────────
def generate_synthetic(X_real, n_pos=5, n_neg=6):
    sigma = X_real.std(axis=0) + 1e-6
    pos, neg = [], []
    for row in X_real:
        for _ in range(n_pos):
            pos.append(np.clip(row + np.random.uniform(-sigma, sigma), 0, None))
        for _ in range(n_neg):
            neg.append(np.random.uniform(1.0, 300.0, size=X_real.shape[1]))
    X_all = np.vstack([X_real, np.array(pos), np.array(neg)])
    y_all = np.concatenate([
        np.ones(len(X_real)+len(pos)), np.zeros(len(neg))])
    return X_all, y_all

X_train, y_train = generate_synthetic(X_train_real, 5, 6)
X_test,  y_test  = generate_synthetic(X_test_real,  5, 6)
print(f'Train: {X_train.shape}  auténticos={int(y_train.sum())}  falsos={int((y_train==0).sum())}')
print(f'Test : {X_test.shape}   auténticos={int(y_test.sum())}   falsos={int((y_test==0).sum())}')

In [ ]:
# ── Tabla 1 · Muestra de vectores de características ─────────────────────────
rows = []
for i, (v, l) in enumerate(zip(X_train[:8], y_train[:8])):
    r = {f'EE{j+1}': int(v[j]) for j in range(5)}
    r.update({'...':'···', 'EE54':int(v[53]),
              'Clase':'Auténtica' if l==1 else 'Falsa'})
    rows.append(r)
df_t = pd.DataFrame(rows, index=[f'Sig {i+1}' for i in range(8)])
print('Tabla 1 · Patrones de Conocimiento para el Aprendizaje')
display(df_t)

In [ ]:
# ── Fig. 9 · Análisis exploratorio ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
mask_pos = y_train == 1
mu_pos = X_train[mask_pos].mean(0);  sd_pos = X_train[mask_pos].std(0)
mu_neg = X_train[~mask_pos].mean(0)
axes[0].plot(mu_pos, label='Auténtica (μ)', color='steelblue')
axes[0].fill_between(range(54), mu_pos-sd_pos, mu_pos+sd_pos,
                     alpha=0.25, color='steelblue')
axes[0].plot(mu_neg, label='Falsa (μ)', color='tomato', linestyle='--')
axes[0].set(xlabel='EE', ylabel='Píxeles encendidos',
            title='Media del vector de características por clase')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].hist(X_train[mask_pos,0],  bins=30, alpha=0.7, label='Auténtica', color='steelblue')
axes[1].hist(X_train[~mask_pos,0], bins=30, alpha=0.7, label='Falsa',     color='tomato')
axes[1].set(title='Distribución de EE1 por clase')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Fig. 9 · Análisis exploratorio del vector de características (54-D)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Etapa 4 · Aprendizaje Supervisado
### Reconocimiento de Patrones de Firmas en Cheques Off-Line

Se entrenan y comparan **cuatro clasificadores**:

| # | Clasificador | Arquitectura / parámetros |
|---|---|---|
| 1 | **BPNN** | 54 entradas → 108 ocultas → 1 salida (sigmoidal) |
| 2 | **SVM** | Kernel RBF, C=10, gamma='scale' |
| 3 | **KNN** | k=5, distancia Euclídea |
| 4 | **Naive Bayes** | Gaussiano |

In [ ]:
# ── Normalización ─────────────────────────────────────────────────────────────
scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)
y_tr    = y_train.astype(int)
y_te    = y_test.astype(int)
print('Normalización StandardScaler ✔')

In [ ]:
# ── Entrenamiento de los cuatro clasificadores ────────────────────────────────
classifiers = {
    'BPNN'        : MLPClassifier(hidden_layer_sizes=(108,), activation='logistic',
                                   solver='adam', max_iter=500, random_state=42),
    'SVM (RBF)'   : SVC(kernel='rbf', C=10, gamma='scale',
                        probability=True, random_state=42),
    'KNN (k=5)'   : KNeighborsClassifier(n_neighbors=5, metric='euclidean'),
    'Naive Bayes' : GaussianNB(),
}
for name, clf in classifiers.items():
    clf.fit(X_tr_sc, y_tr)
    print(f'{name:<18} ✔ entrenado')

In [ ]:
# ── Evaluación ────────────────────────────────────────────────────────────────
results = {}
for name, clf in classifiers.items():
    y_pred   = clf.predict(X_te_sc)
    cv       = cross_val_score(clf, X_tr_sc, y_tr, cv=5, scoring='accuracy')
    results[name] = dict(y_pred=y_pred, accuracy=accuracy_score(y_te, y_pred),
                         cv_mean=cv.mean(), cv_std=cv.std())
    print(f'{name:<18}  Acc={results[name]["accuracy"]:.4f}  '
          f'CV={cv.mean():.4f}±{cv.std():.4f}')

In [ ]:
# ── Fig. 10 · Matrices de confusión ──────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (name, res) in zip(axes.flatten(), results.items()):
    ConfusionMatrixDisplay(
        confusion_matrix(y_te, res['y_pred']),
        display_labels=['Falsa','Auténtica']
    ).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}  —  Acc={res["accuracy"]:.4f}',
                 fontsize=11, fontweight='bold')
plt.suptitle('Fig. 10 · Matrices de Confusión — Clasificadores Off-Line',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ── Reportes detallados ───────────────────────────────────────────────────────
for name, res in results.items():
    print(f'\n══ {name} ══')
    print(classification_report(y_te, res['y_pred'],
                                target_names=['Falsa','Auténtica'], digits=4))

In [ ]:
# ── Fig. 11 · Curvas ROC ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
colors  = ['steelblue','tomato','seagreen','darkorchid']
for (name, clf), c in zip(classifiers.items(), colors):
    proba = (clf.predict_proba(X_te_sc)[:,1]
             if hasattr(clf,'predict_proba')
             else clf.decision_function(X_te_sc))
    fpr, tpr, _ = roc_curve(y_te, proba)
    ax.plot(fpr, tpr, color=c, lw=2, label=f'{name}  AUC={auc(fpr,tpr):.4f}')
ax.plot([0,1],[0,1],'k--',lw=1.2,label='Aleatorio')
ax.set(xlim=[0,1], ylim=[0,1.05], xlabel='FPR', ylabel='TPR',
       title='Fig. 11 · Curvas ROC — Comparación de Clasificadores')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Fig. 12 · Tabla y gráfica comparativa ────────────────────────────────────
rows = [{
    'Clasificador' : name,
    'Acc. (Test)'  : f"{res['accuracy']:.4f}",
    'CV-5 Media'   : f"{res['cv_mean']:.4f}",
    'CV-5 Std'     : f"{res['cv_std']:.4f}",
    'Precisión'    : f"{precision_score(y_te, res['y_pred']):.4f}",
    'Recall'       : f"{recall_score(y_te, res['y_pred']):.4f}",
    'F1'           : f"{f1_score(y_te, res['y_pred']):.4f}",
} for name, res in results.items()]

df_res = pd.DataFrame(rows).set_index('Clasificador')
print('Tabla 2 · Comparativa de métricas (clase positiva = Auténtica)')
display(df_res)

df_res[['Acc. (Test)','Precisión','Recall','F1']].astype(float).plot(
    kind='bar', figsize=(11,5), colormap='Set2', edgecolor='white', width=0.75)
plt.title('Fig. 12 · Comparación de métricas por clasificador',
          fontsize=12, fontweight='bold')
plt.ylim(0, 1.12); plt.ylabel('Valor')
plt.legend(loc='lower right', fontsize=9)
plt.xticks(rotation=20, ha='right')
plt.grid(axis='y', alpha=0.35)
plt.tight_layout(); plt.show()

In [ ]:
# ── Fig. 13 · Curva de pérdida BPNN ──────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(classifiers['BPNN'].loss_curve_, color='steelblue', lw=2)
plt.title('Fig. 13 · Curva de pérdida — BPNN', fontsize=12, fontweight='bold')
plt.xlabel('Época'); plt.ylabel('Cross-Entropy Loss')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
## Demo · Verificación de una firma en un cheque nuevo

Una vez entrenado el sistema, el flujo completo para verificar cualquier cheque es:

In [ ]:
# ── Demo: verificar firma en un cheque nuevo ──────────────────────────────────
# Sustituye NEW_CHECK_PATH por la ruta a cualquier cheque real.
NEW_CHECK_PATH = test_X_paths[0]

# 1. Extraer firma automáticamente con la U-Net
firma = extract_signature_from_check(NEW_CHECK_PATH, model)

# 2. Extraer vector de características
vec   = extract_features(firma, ALL_SES).reshape(1, -1)
vec_sc = scaler.transform(vec)

# 3. Clasificar con todos los modelos
print('Resultado de verificación:')
print('─' * 45)
for name, clf in classifiers.items():
    pred  = clf.predict(vec_sc)[0]
    proba = clf.predict_proba(vec_sc)[0] if hasattr(clf,'predict_proba') else None
    lbl   = '✅ AUTÉNTICA' if pred==1 else '❌ FALSIFICACIÓN'
    conf  = f'  (conf. {max(proba)*100:.1f}%)' if proba is not None else ''
    print(f'{name:<18} → {lbl}{conf}')

# 4. Mostrar
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(np.array(Image.open(NEW_CHECK_PATH).convert('RGB')))
axes[0].set_title('Cheque de entrada', fontsize=10); axes[0].axis('off')
axes[1].imshow(firma, cmap='gray')
axes[1].set_title('Firma extraída por U-Net', fontsize=10); axes[1].axis('off')
plt.suptitle('Demo · Verificación automática desde cheque real',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Conclusiones

Se implementó un sistema completo y totalmente automático de **verificación de firmas off-line en cheques** con las siguientes etapas:

1. **Etapa 1 — Selección de datos:** Se cargaron los pares `(cheque completo, firma ground truth)` del dataset estructurado. Las imágenes en `y` sirven exclusivamente como supervisión para la red de segmentación.

2. **Etapa 2 — Pre-procesamiento con U-Net:** Se entrenó una U-Net ligera para localizar y extraer automáticamente la región de la firma desde cualquier cheque real. La función de pérdida combinada BCE+Dice es robusta ante el desequilibrio entre la región de firma y el resto del cheque. A partir de este punto, el sistema no requiere intervención manual: dado cualquier cheque, la red extrae la firma.

3. **Etapa 3 — Extracción de características:** Mediante erosión morfológica con 54 elementos estructurantes (36 curvos + 18 rectos), se construyó un vector de 54 componentes por firma, que captura la distribución de trazos en la imagen. Los patrones sintéticos positivos y negativos aumentan el conjunto de entrenamiento.

4. **Etapa 4 — Aprendizaje supervisado:** Los cuatro clasificadores (BPNN 54→108→1, SVM-RBF, KNN k=5 y Naive Bayes) muestran que los modelos no lineales —especialmente BPNN y SVM— obtienen las mejores métricas, al capturar la frontera de decisión compleja entre firma auténtica y falsificación.

**Trabajo futuro:** Aumentar el dataset de firmas, usar transfer learning en la U-Net (e.g., ResNet encoder), combinar características morfológicas con HOG o LBP, y explorar redes siamesas para comparación directa de pares de firmas.

---

## Referencias

Plamondon, R., & Srihari, S. N. (2000). Online and off-line handwriting recognition: A comprehensive survey. *IEEE Transactions on Pattern Analysis and Machine Intelligence, 22*(1), 63–84. https://doi.org/10.1109/34.824821

Lee, L.-L., Berger, T., & Aviczer, E. (1996). Reliable on-line human signature verification systems. *IEEE Transactions on Pattern Analysis and Machine Intelligence, 18*(6), 643–647. https://doi.org/10.1109/34.506415

Ronneberger, O., Fischer, P., & Brox, T. (2015). U-Net: Convolutional networks for biomedical image segmentation. *MICCAI 2015, Lecture Notes in Computer Science, 9351*, 234–241. https://doi.org/10.1007/978-3-319-24574-4_28

Marsland, S. (2009). *Machine learning: An algorithmic perspective*. CRC Press.

Gonzalez, R. C., & Woods, R. E. (2018). *Digital image processing* (4.ª ed.). Pearson.

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.